# Case Palavritas - the news
## Analista de Dados Produto & Growth

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from src.load import load_sessions, load_attempts, load_user_profile, save_cleaned
from src.cleaning import rodar_diagnostico
from src.analysis import (
    retencao_por_resultado, retencao_por_streak, retencao_por_hora,
    retencao_por_device, retencao_por_newsletter, dificuldade_por_palavra,
    juntar_sessions_profile, retencao_por_perfil, padrao_tentativas,
    matriz_correlacao, tabela_features, analise_coorte, curva_retencao,
    retencao_por_usuario, segmentar_usuarios
)
from src.statistics import (
    teste_z_proporcoes, ranking_correlacao, teste_anova, teste_t,
    ic_proporcao, correcao_bonferroni
)
from src.plots import (
    grafico_retencao_resultado, grafico_retencao_streak, grafico_retencao_hora,
    grafico_palavras, grafico_heatmap, grafico_perfil, grafico_ranking,
    grafico_curva_retencao, grafico_coorte
)

## 1. Carregamento e Diagnostico

In [2]:
sessions, attempts, profile = rodar_diagnostico()

DIAGNOSTICO - palavritas_sessions
Linhas originais: 41157
session_id duplicados: 1198
Datas com formato inconsistente: 0
Tentativas fora de 1-6: 93 valores=[np.int64(0), np.int64(7), np.int64(8)]
Resultados invalidos: 63
Devices brutos: {'ios': 22426, 'android': 18731}
Horas invalidas: 0
Tempo min=-5 max=480 media=269.5
Outliers de tempo (P1=63 P99=476): 746
played_next_day: {False: 32015, True: 9142}
active_d30: {False: 28015, True: 13142}
newsletter_open: {False: 33238, True: 7919}
streak_day: {1: 32032, 2: 6963, 3: 1627, 4: 384, 5: 108, 6: 31, 7: 9, 8: 3}

DIAGNOSTICO - palavritas_attempts
Linhas originais: 147270
attempt_number fora de 1-6: 41
correct_letters fora de 0-5: 0
correct_positions fora de 0-5: 0
letters + positions > 5: 12
Sessions com attempts: 40069

DIAGNOSTICO - user_profile
Linhas originais: 800
user_id duplicados: 0
Nulos por coluna:
{'age_range': 117, 'city': 297, 'salary_range': 193}
age_range: ['18-24', '25-34', '35-44', '45+']
Estados por extenso: ['São Paulo',


RESUMO POS-LIMPEZA: sessions=39849 attempts=147217 profile=800


In [3]:
sessions.head()

,session_id,user_id,word,word_date,attempts,result,time_to_complete_sec,device,session_hour,streak_day,played_next_day,newsletter_open_before_game,active_d30
0,ab38635a07ede4246661,697e9e150e91bb76,TEMPO,2026-02-08,1,win,449,Android,20,2,False,False,False
1,67d40a22cba1a16fd2c5,d7f63dda905adaec,FALHA,2026-05-08,6,lose,232,iOS,19,1,True,False,False
2,3635c5707f62ea022423,6220de621fef79b8,FALHA,2026-04-18,3,win,65,iOS,7,2,False,False,False
3,27243d4c7b1669f71ba7,3aa2f6a86d7fe06c,PRATO,2025-12-22,6,lose,404,Android,23,2,False,False,True
4,cd21162dd072f066c160,97af736a7f2aa637,IDEIA,2026-05-28,1,win,222,Android,6,2,False,True,False


In [4]:
attempts.head()

,session_id,attempt_number,guess,correct_letters,correct_positions
0,ab38635a07ede4246661,1,TEMPO,0,5
1,67d40a22cba1a16fd2c5,1,NOVAS,1,0
2,67d40a22cba1a16fd2c5,2,OUVIR,0,0
3,67d40a22cba1a16fd2c5,3,NOVAS,1,0
4,67d40a22cba1a16fd2c5,4,VAPOR,0,1


In [5]:
profile.head()

,user_id,age_range,state,city,salary_range,job_role,sector,company_size,orders_food_delivery,food_delivery_freq_week,food_delivery_platform,primary_device,plays_other_word_games,typical_play_time,newsletter_subscriber
0,fe2e4f159653f26f,Nao informado,MG,Nao informado,R$4k-R$6k,Consultor,finanças,pequena,True,5,Rappi,Android,True,afternoon,True
1,4d89d721da20cbc3,35-44,MG,Rio de Janeiro,R$4k-R$6k,analista,educação,multinacional,True,3,Ambos,iOS,True,morning,True
2,91bf47f1274e23a8,25-34,PB,Nao informado,Nao informado,Coordenador,tech,pequena,True,3,Nenhum,Android,True,morning,True
3,4efe31b67538c37f,25-34,PB,Nao informado,até R$2k,Gerente,finanças,grande,True,5,Nenhum,iOS,False,afternoon,False
4,d6af92f33a83c685,18-24,RN,Belo Horizonte,até R$2k,consultor,outros,pequena,False,0,Nenhum,Android,True,evening,False


## 2. Retencao por Resultado (win/lose) com IC 95%

In [6]:
r_resultado = retencao_por_resultado(sessions)
r_resultado

,result,sessoes,d1_pct,d30_pct,media_tentativas,media_tempo
0,lose,15806,0.224915,0.308364,6.000000,269.804758
1,win,24043,0.219107,0.326623,1.995217,269.995674


In [7]:
grafico_retencao_resultado(r_resultado)

Salvo: C:\Users\felip\OneDrive\Documentos\CodingThings\case-data-analyst-the-news\notebooks\..\reports\figures\retencao_resultado.png


In [8]:
wins_d1 = sessions[sessions['result'] == 'win']['played_next_day'].sum()
wins_total = (sessions['result'] == 'win').sum()
loses_d1 = sessions[sessions['result'] == 'lose']['played_next_day'].sum()
loses_total = (sessions['result'] == 'lose').sum()

z = teste_z_proporcoes(wins_d1, wins_total, loses_d1, loses_total)
ic_win = ic_proporcao(wins_d1, wins_total)
ic_lose = ic_proporcao(loses_d1, loses_total)

print('Teste Z - resultado vs D+1:')
print(f"  Win D+1: {z['prop_a']:.2%}  IC95=[{ic_win['ic_inferior']:.2%}, {ic_win['ic_superior']:.2%}]")
print(f"  Lose D+1: {z['prop_b']:.2%} IC95=[{ic_lose['ic_inferior']:.2%}, {ic_lose['ic_superior']:.2%}]")
print(f"  z={z['z']:.3f}, p={z['p_valor']:.4f}, significativo={z['significativo']}")

Teste Z - resultado vs D+1:
  Win D+1: 21.91%  IC95=[21.39%, 22.43%]
  Lose D+1: 22.49% IC95=[21.84%, 23.14%]
  z=-1.366, p=0.1720, significativo=False


## 3. Retencao por Streak

In [9]:
r_streak = retencao_por_streak(sessions)
r_streak

,streak_day,sessoes,d1_pct,d30_pct
0,1,31032,0.216776,0.319541
1,2,6720,0.234375,0.318155
2,3,1576,0.237944,0.314086
3,4,374,0.275401,0.339572
4,5,104,0.298077,0.355769
5,6,31,0.290323,0.354839
6,7,9,0.333333,0.333333
7,8,3,0.000000,0.000000


In [10]:
grafico_retencao_streak(r_streak)

Salvo: C:\Users\felip\OneDrive\Documentos\CodingThings\case-data-analyst-the-news\notebooks\..\reports\figures\retencao_streak.png


## 4. Retencao por Hora do Dia

In [11]:
r_hora = retencao_por_hora(sessions)
r_hora

,session_hour,sessoes,d1_pct,d30_pct,win_rate
0,6,4145,0.211580,0.363571,0.759710
1,7,4252,0.228833,0.384760,0.754233
2,8,4160,0.219471,0.378365,0.760817
3,12,1386,0.214286,0.272727,0.510823
4,13,1401,0.227695,0.291221,0.551035
5,14,1385,0.238989,0.280144,0.541516
6,15,1412,0.247167,0.342776,0.530453
7,16,1348,0.199555,0.287834,0.516320
8,17,1373,0.226511,0.310269,0.522214
9,18,2654,0.212133,0.274303,0.533157


In [12]:
grafico_retencao_hora(r_hora)

Salvo: C:\Users\felip\OneDrive\Documentos\CodingThings\case-data-analyst-the-news\notebooks\..\reports\figures\retencao_hora.png


## 5. Retencao por Device

In [13]:
r_device = retencao_por_device(sessions)
r_device

,device,sessoes,d1_pct,d30_pct,win_rate
0,Android,18111,0.219756,0.318149,0.631053
1,iOS,21738,0.222790,0.320407,0.580274


In [14]:
android_d30 = sessions[(sessions['device'] == 'Android')]['active_d30'].sum()
android_n = (sessions['device'] == 'Android').sum()
ios_d30 = sessions[(sessions['device'] == 'iOS')]['active_d30'].sum()
ios_n = (sessions['device'] == 'iOS').sum()

z = teste_z_proporcoes(android_d30, android_n, ios_d30, ios_n)
print('Teste Z - device vs D30:')
print(f"  Android D30: {z['prop_a']:.2%}, iOS D30: {z['prop_b']:.2%}")
print(f"  z={z['z']:.3f}, p={z['p_valor']:.4f}, significativo={z['significativo']}")

Teste Z - device vs D30:
  Android D30: 31.81%, iOS D30: 32.04%
  z=-0.481, p=0.6303, significativo=False


## 6. Newsletter e Retencao

In [15]:
r_news = retencao_por_newsletter(sessions)
r_news

,newsletter_open_before_game,sessoes,d1_pct,d30_pct,win_rate
0,False,32198,0.222840,0.305423,0.563793
1,True,7651,0.215397,0.378121,0.769834


In [16]:
open_d1 = sessions[sessions['newsletter_open_before_game'] == True]['played_next_day'].sum()
open_n = (sessions['newsletter_open_before_game'] == True).sum()
no_open_d1 = sessions[sessions['newsletter_open_before_game'] == False]['played_next_day'].sum()
no_open_n = (sessions['newsletter_open_before_game'] == False).sum()

z = teste_z_proporcoes(open_d1, open_n, no_open_d1, no_open_n)
ic_open = ic_proporcao(open_d1, open_n)
ic_no = ic_proporcao(no_open_d1, no_open_n)

print('Teste Z - newsletter vs D+1:')
print(f"  Abriu: {z['prop_a']:.2%} IC95=[{ic_open['ic_inferior']:.2%}, {ic_open['ic_superior']:.2%}]")
print(f"  Nao abriu: {z['prop_b']:.2%} IC95=[{ic_no['ic_inferior']:.2%}, {ic_no['ic_superior']:.2%}]")
print(f"  z={z['z']:.3f}, p={z['p_valor']:.4f}, significativo={z['significativo']}")

Teste Z - newsletter vs D+1:
  Abriu: 21.54% IC95=[20.62%, 22.46%]
  Nao abriu: 22.28% IC95=[21.83%, 22.74%]
  z=-1.410, p=0.1587, significativo=False


## 7. Dificuldade por Palavra

In [17]:
r_palavras = dificuldade_por_palavra(sessions)
r_palavras.head(15)

,word,sessoes,win_rate,media_tentativas,d1_pct,d30_pct
5,DANÇA,937,0.440768,4.249733,0.226254,0.319104
3,CIÚME,912,0.447368,4.231360,0.232456,0.330044
19,PAZÃO,1764,0.449546,4.195011,0.239229,0.328231
18,ORAÇÃ,837,0.457587,4.163680,0.213859,0.314217
10,HERÓI,2602,0.458109,4.162567,0.228286,0.327056
16,NIXÃO,906,0.461369,4.134658,0.236203,0.337748
6,FALHA,1783,0.621425,3.498598,0.220415,0.318564
8,FRACO,905,0.622099,3.472928,0.193370,0.309392
26,TEMPO,1776,0.623874,3.520833,0.204392,0.323198
2,CASCA,1763,0.624504,3.511628,0.228020,0.319342


In [18]:
grafico_palavras(r_palavras)

Salvo: C:\Users\felip\OneDrive\Documentos\CodingThings\case-data-analyst-the-news\notebooks\..\reports\figures\palavras_dificuldade.png


## 8. Perfil dos Jogadores

In [19]:
merged = juntar_sessions_profile(sessions, profile)
print(f'Sessoes com perfil: {len(merged)} de {len(sessions)}')
print(f'Jogadores com perfil: {merged["user_id"].nunique()} de {sessions["user_id"].nunique()}')

Sessoes com perfil: 26691 de 39849
Jogadores com perfil: 800 de 1200


In [20]:
for var in ['salary_range', 'sector', 'age_range', 'typical_play_time']:
    print(f'\n--- {var} ---')
    r = retencao_por_perfil(merged, var)
    print(r.head(8).to_string())
    grafico_perfil(r, var, f'Retencao por {var}')


--- salary_range ---
     salary_range  sessoes    d1_pct   d30_pct
0   Nao informado     6406  0.219482  0.327193
2       R$4k-R$6k     5387  0.219046  0.323557
1       R$2k-R$4k     5082  0.231602  0.326643
3      R$6k-R$10k     3673  0.215628  0.312007
5        até R$2k     3309  0.229374  0.329707
4  acima de R$10k     2834  0.227594  0.304869


Salvo: C:\Users\felip\OneDrive\Documentos\CodingThings\case-data-analyst-the-news\notebooks\..\reports\figures\perfil_salary_range.png

--- sector ---
      sector  sessoes    d1_pct   d30_pct
6       tech     6095  0.232978  0.323216
4     outros     5066  0.217726  0.315239
2   finanças     3766  0.216144  0.319703
1   educação     3381  0.236025  0.337770
5      saúde     2547  0.217118  0.309776
7     varejo     2406  0.227348  0.330840
3  marketing     2030  0.212315  0.324138
0    direito     1400  0.209286  0.317143
Salvo: C:\Users\felip\OneDrive\Documentos\CodingThings\case-data-analyst-the-news\notebooks\..\reports\figures\perfil_sector.png

--- age_range ---
       age_range  sessoes    d1_pct   d30_pct
1          25-34     9640  0.225519  0.321680
2          35-44     6634  0.216008  0.323334
4  Nao informado     3928  0.220978  0.317974
3            45+     3267  0.238751  0.326599
0          18-24     3222  0.218498  0.322160


Salvo: C:\Users\felip\OneDrive\Documentos\CodingThings\case-data-analyst-the-news\notebooks\..\reports\figures\perfil_age_range.png

--- typical_play_time ---
  typical_play_time  sessoes    d1_pct   d30_pct
2           morning     8536  0.227624  0.378163
1           evening     7216  0.215493  0.290050
0         afternoon     5704  0.231767  0.304173
3             night     5235  0.217574  0.294938
Salvo: C:\Users\felip\OneDrive\Documentos\CodingThings\case-data-analyst-the-news\notebooks\..\reports\figures\perfil_typical_play_time.png


In [21]:
print('ANOVA - salary_range vs D30:')
r = teste_anova(merged, 'salary_range', 'active_d30')
print(f"  F={r['f']:.3f}, p={r['p_valor']:.4f}, significativo={r['significativo']}")

print('ANOVA - sector vs D30:')
r = teste_anova(merged, 'sector', 'active_d30')
print(f"  F={r['f']:.3f}, p={r['p_valor']:.4f}, significativo={r['significativo']}")

ANOVA - salary_range vs D30:
  F=1.547, p=0.1715, significativo=False
ANOVA - sector vs D30:
  F=1.119, p=0.3474, significativo=False


## 9. Curva de Retencao Global

In [22]:
curva = curva_retencao(sessions)
curva[curva['dia'].isin([1, 3, 7, 14, 30])]

,dia,ativos,retencao
1,1,221,0.184167
3,3,223,0.185833
7,7,209,0.174167
14,14,208,0.173333
30,30,229,0.190833


In [23]:
grafico_curva_retencao(curva)

Salvo: C:\Users\felip\OneDrive\Documentos\CodingThings\case-data-analyst-the-news\notebooks\..\reports\figures\curva_retencao.png


## 10. Analise de Coorte Semanal

In [24]:
cohort = analise_coorte(sessions)
cohort.head(20)

,cohort_week,week_number,ativos,tamanho,retencao
0,2025-12-01/2025-12-07,0,848,848,1.000000
1,2025-12-01/2025-12-07,1,653,848,0.770047
2,2025-12-01/2025-12-07,2,658,848,0.775943
3,2025-12-01/2025-12-07,3,644,848,0.759434
4,2025-12-01/2025-12-07,4,649,848,0.765330
5,2025-12-01/2025-12-07,5,636,848,0.750000
6,2025-12-01/2025-12-07,6,659,848,0.777123
7,2025-12-01/2025-12-07,7,658,848,0.775943
8,2025-12-01/2025-12-07,8,621,848,0.732311
9,2025-12-01/2025-12-07,9,657,848,0.774764


In [25]:
grafico_coorte(cohort)

Salvo: C:\Users\felip\OneDrive\Documentos\CodingThings\case-data-analyst-the-news\notebooks\..\reports\figures\coorte.png


## 11. Segmentacao de Usuarios

In [26]:
user_metrics = retencao_por_usuario(sessions)
user_metrics = segmentar_usuarios(user_metrics)
user_metrics['segmento'].value_counts()

segmento
heavy     1182
medium      18
Name: count, dtype: int64

In [27]:
user_metrics.groupby('segmento').agg(
    usuarios=('user_id', 'count'),
    d1_medio=('d1_medio', 'mean'),
    d30_medio=('d30_medio', 'mean'),
    win_rate_medio=('win_rate', 'mean'),
    streak_max_medio=('streak_max', 'mean'),
).round(3)

,usuarios,d1_medio,d30_medio,win_rate_medio,streak_max_medio
segmento,,,,,
heavy,1182,0.183,0.302,0.603,2.815
medium,18,0.011,0.256,0.700,1.056


## 12. Correlacoes e Ranking

In [28]:
corr = matriz_correlacao(sessions)
corr

,attempts,time_to_complete_sec,session_hour,streak_day,played_next_day,active_d30,newsletter_open_before_game,win
attempts,1.000000,-0.001609,0.177398,0.001730,0.008101,-0.019260,-0.156658,-0.951464
time_to_complete_sec,-0.001609,1.000000,-0.004336,-0.008629,0.007274,0.002544,0.005796,0.000766
session_hour,0.177398,-0.004336,1.000000,0.004887,0.002739,-0.070705,-0.639328,-0.188926
streak_day,0.001730,-0.008629,0.004887,1.000000,0.023326,0.001057,-0.001803,-0.002926
played_next_day,0.008101,0.007274,0.002739,0.023326,1.000000,0.399549,-0.007061,-0.006842
active_d30,-0.019260,0.002544,-0.070705,0.001057,0.399549,1.000000,0.061415,0.019159
newsletter_open_before_game,-0.156658,0.005796,-0.639328,-0.001803,-0.007061,0.061415,1.000000,0.165891
win,-0.951464,0.000766,-0.188926,-0.002926,-0.006842,0.019159,0.165891,1.000000


In [29]:
grafico_heatmap(corr)

Salvo: C:\Users\felip\OneDrive\Documentos\CodingThings\case-data-analyst-the-news\notebooks\..\reports\figures\heatmap.png


In [30]:
ranking_d1 = ranking_correlacao(corr, 'played_next_day')
ranking_d1

,variavel,correlacao,abs_corr
0,active_d30,0.399549,0.399549
1,streak_day,0.023326,0.023326
2,attempts,0.008101,0.008101
3,time_to_complete_sec,0.007274,0.007274
4,newsletter_open_before_game,-0.007061,0.007061
5,win,-0.006842,0.006842
6,session_hour,0.002739,0.002739


In [31]:
ranking_d30 = ranking_correlacao(corr, 'active_d30')
ranking_d30

,variavel,correlacao,abs_corr
0,played_next_day,0.399549,0.399549
1,session_hour,-0.070705,0.070705
2,newsletter_open_before_game,0.061415,0.061415
3,attempts,-0.019260,0.019260
4,win,0.019159,0.019159
5,time_to_complete_sec,0.002544,0.002544
6,streak_day,0.001057,0.001057


## 13. Features com Perfil

In [32]:
features = tabela_features(sessions, profile)
corr_full = features.corr()
ranking_features_d1 = ranking_correlacao(corr_full, 'jogou_dia_seguinte')
ranking_features_d1.head(12)

C:\Users\felip\OneDrive\Documentos\CodingThings\case-data-analyst-the-news\notebooks\..\src\analysis.py:127: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "joga_outros": df["plays_other_word_games"].fillna(False).astype(int),
C:\Users\felip\OneDrive\Documentos\CodingThings\case-data-analyst-the-news\notebooks\..\src\analysis.py:128: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "assinante_news": df["newsletter_subscriber"].fillna(False).astype(int),
C:\Users\felip\OneDrive\Documentos\CodingThings\case-data-analyst-the-news\notebooks\..\src\analysis.py:12

,variavel,correlacao,abs_corr
0,ativo_d30,0.399549,0.399549
1,streak,0.023326,0.023326
2,pede_comida,0.012898,0.012898
3,assinante_news,0.011680,0.011680
4,horario_afternoon,0.010195,0.010195
5,tentativas,0.008101,0.008101
6,horario_morning,0.007813,0.007813
7,tempo_total,0.007274,0.007274
8,abriu_newsletter,-0.007061,0.007061
9,vitoria,-0.006842,0.006842


In [33]:
ranking_features_d30 = ranking_correlacao(corr_full, 'ativo_d30')
ranking_features_d30.head(12)

,variavel,correlacao,abs_corr
0,jogou_dia_seguinte,0.399549,0.399549
1,hora,-0.070705,0.070705
2,horario_morning,0.065827,0.065827
3,abriu_newsletter,0.061415,0.061415
4,horario_evening,-0.029583,0.029583
5,horario_night,-0.020388,0.020388
6,tentativas,-0.019260,0.019260
7,vitoria,0.019159,0.019159
8,horario_afternoon,-0.013332,0.013332
9,assinante_news,0.011325,0.011325


In [34]:
grafico_ranking(ranking_features_d1, 'O que mais se associa com jogar no dia seguinte?')

Salvo: C:\Users\felip\OneDrive\Documentos\CodingThings\case-data-analyst-the-news\notebooks\..\reports\figures\ranking.png


## 14. Protocolo de Tentativas

In [35]:
padrao = padrao_tentativas(attempts, sessions)
padrao

,result,attempt_number,media_corretas,media_letras,contagem
0,lose,1,0.556095,0.841151,16267
1,lose,2,0.553062,0.833005,16264
2,lose,3,0.556423,0.838783,16270
3,lose,4,0.550876,0.835967,16265
4,lose,5,0.553499,0.837720,16262
5,lose,6,0.545589,0.839840,16265
6,win,1,2.058223,0.566278,24767
7,win,2,2.806267,0.418132,16435
8,win,3,4.999510,0.000000,8165
9,win,4,2.500000,1.000000,4


## 15. Salvar dados limpos

In [36]:
save_cleaned(sessions, attempts, profile)
print('Notebook executado com sucesso!')

Salvo em C:\Users\felip\OneDrive\Documentos\CodingThings\case-data-analyst-the-news\notebooks\..\data\processed
Notebook executado com sucesso!
